# Project · Diagnose from measurements: generative vs discriminative, then decide

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). 569 breast-tissue samples, 30 measurements each, labelled malignant or benign (scikit-learn's breast cancer dataset). You'll build three classifiers from scratch: Gaussian naive Bayes, LDA and logistic regression. Then you'll compare them the way a statistician would: ROC curves, calibration (do the probabilities mean what they say?), a decision rule that weighs a missed cancer as 10× worse than a false alarm, and learning curves. Uses Murphy ch. 5, 9 and 10.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Load, split, standardise

Load the data, make y = 1 for malignant, split 80/20 (stratified by hand with a fixed seed), and standardise with training statistics. How unbalanced are the classes?

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_breast_cancer
d = load_breast_cancer()
X, y = d.data, (d.target == 0).astype(float)          # 1 = malignant
rng = np.random.default_rng(0)
tr, te = [], []
for c in (0, 1):                                      # stratified split: 80/20 within each class
    idx = rng.permutation(np.where(y == c)[0]); cut = int(0.8 * len(idx))
    tr += list(idx[:cut]); te += list(idx[cut:])
tr, te = np.array(tr), np.array(te)
mu, sd = X[tr].mean(0), X[tr].std(0)
Xtr, Xte = ...  # TODO: training statistics only
ytr, yte = y[tr], y[te]
print(Xtr.shape, Xte.shape, f"malignant share: train {ytr.mean():.3f}, test {yte.mean():.3f}")

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_breast_cancer
d = load_breast_cancer()
X, y = d.data, (d.target == 0).astype(float)          # 1 = malignant
rng = np.random.default_rng(0)
tr, te = [], []
for c in (0, 1):                                      # stratified split: 80/20 within each class
    idx = rng.permutation(np.where(y == c)[0]); cut = int(0.8 * len(idx))
    tr += list(idx[:cut]); te += list(idx[cut:])
tr, te = np.array(tr), np.array(te)
mu, sd = X[tr].mean(0), X[tr].std(0)
Xtr, Xte = (X[tr] - mu) / sd, (X[te] - mu) / sd  # training statistics only
ytr, yte = y[tr], y[te]
print(Xtr.shape, Xte.shape, f"malignant share: train {ytr.mean():.3f}, test {yte.mean():.3f}")

<details><summary><b>Answer and explanation</b></summary>

454 training and 115 test samples, about 37% malignant in each (the stratified split keeps the ratio). It's unbalanced enough that 'always benign' would score 63%, so that is the baseline every model must clearly beat.

Expected output:
```
(454, 30) (115, 30) malignant share: train 0.372, test 0.374
```

</details>

## Milestone 2 · Gaussian naive Bayes from scratch

Fit one mean and one variance per feature per class, plus the class prior. Score a point by log prior + Σ log N(xⱼ | μ, σ²), and turn the two scores into P(malignant | x) with a softmax. Report test accuracy.

In [ ]:
# Your turn
def nb_fit(X, y):
    return [(np.mean(y == c), X[y == c].mean(0), X[y == c].var(0) + 1e-6) for c in (0, 1)]
def nb_proba(model, X):
    s = ...  # TODO: log p(c) + sum_j log N(x_j | mu, var)
    s -= s.max(1, keepdims=True); e = np.exp(s)
    return e[:, 1] / e.sum(1)
nb = nb_fit(Xtr, ytr); p_nb = nb_proba(nb, Xte)
print(f"naive Bayes test accuracy {np.mean((p_nb > 0.5) == yte):.3f}")

In [ ]:
# @title Solution 2 (double-click to show the code)
def nb_fit(X, y):
    return [(np.mean(y == c), X[y == c].mean(0), X[y == c].var(0) + 1e-6) for c in (0, 1)]
def nb_proba(model, X):
    s = np.stack([np.log(pr) - 0.5 * np.sum(np.log(2 * np.pi * v) + (X - m) ** 2 / v, 1) for pr, m, v in model], 1)  # log p(c) + sum_j log N(x_j | mu, var)
    s -= s.max(1, keepdims=True); e = np.exp(s)
    return e[:, 1] / e.sum(1)
nb = nb_fit(Xtr, ytr); p_nb = nb_proba(nb, Xte)
print(f"naive Bayes test accuracy {np.mean((p_nb > 0.5) == yte):.3f}")

<details><summary><b>Answer and explanation</b></summary>

Naive Bayes gets ≈ 90% test accuracy, well above the 63% baseline, but it's the weakest of the three. The 30 features come in families (radius, perimeter and area of the same cell are nearly the same measurement), so the 'independent given the class' assumption is badly wrong: naive Bayes counts the same evidence three times (Murphy §9.3).

Expected output:
```
naive Bayes test accuracy 0.904
```

</details>

## Milestone 3 · LDA from scratch

Now model each class as a multivariate Gaussian with one shared covariance (add 1e−3·I so it inverts). The log-odds is linear: wᵀx + b with w = Σ⁻¹(μ₁ − μ₀). Report test accuracy.

In [ ]:
# Your turn
m0, m1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
R = np.concatenate([Xtr[ytr == 0] - m0, Xtr[ytr == 1] - m1])
S = R.T @ R / len(R) + 1e-3 * np.eye(Xtr.shape[1])                 # shared covariance
w_lda = ...  # TODO: w = S^-1 (mu1 - mu0)
b_lda = -0.5 * (m1 @ np.linalg.solve(S, m1) - m0 @ np.linalg.solve(S, m0)) + np.log(ytr.mean() / (1 - ytr.mean()))
p_lda = 1 / (1 + np.exp(-(Xte @ w_lda + b_lda)))
print(f"LDA test accuracy {np.mean((p_lda > 0.5) == yte):.3f}")

In [ ]:
# @title Solution 3 (double-click to show the code)
m0, m1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
R = np.concatenate([Xtr[ytr == 0] - m0, Xtr[ytr == 1] - m1])
S = R.T @ R / len(R) + 1e-3 * np.eye(Xtr.shape[1])                 # shared covariance
w_lda = np.linalg.solve(S, m1 - m0)  # w = S^-1 (mu1 - mu0)
b_lda = -0.5 * (m1 @ np.linalg.solve(S, m1) - m0 @ np.linalg.solve(S, m0)) + np.log(ytr.mean() / (1 - ytr.mean()))
p_lda = 1 / (1 + np.exp(-(Xte @ w_lda + b_lda)))
print(f"LDA test accuracy {np.mean((p_lda > 0.5) == yte):.3f}")

<details><summary><b>Answer and explanation</b></summary>

LDA gets ≈ 94%. Modelling the correlations between features with a full shared covariance removes naive Bayes' double counting. The boundary is still a straight line (a weighted sum of features), with weights w = Σ⁻¹(μ₁ − μ₀): Σ⁻¹ down-weights features that are redundant with others (Murphy §9.2).

Expected output:
```
LDA test accuracy 0.939
```

</details>

## Milestone 4 · Logistic regression by gradient descent

Fit logistic regression with a small L2 penalty (λ = 0.01) by gradient descent: 3,000 steps, learning rate 0.5. Report test accuracy.

In [ ]:
# Your turn
A = np.column_stack([np.ones(len(Xtr)), Xtr]); Ate = np.column_stack([np.ones(len(Xte)), Xte])
w = np.zeros(A.shape[1])
for _ in range(3000):
    p = 1 / (1 + np.exp(-A @ w))
    g = ...  # TODO: (prediction - truth) x input, plus weight decay
    w -= 0.5 * g
p_lr = 1 / (1 + np.exp(-Ate @ w))
print(f"logistic regression test accuracy {np.mean((p_lr > 0.5) == yte):.3f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
A = np.column_stack([np.ones(len(Xtr)), Xtr]); Ate = np.column_stack([np.ones(len(Xte)), Xte])
w = np.zeros(A.shape[1])
for _ in range(3000):
    p = 1 / (1 + np.exp(-A @ w))
    g = A.T @ (p - ytr) / len(ytr) + 0.01 * np.r_[0, w[1:]]  # (prediction - truth) x input, plus weight decay
    w -= 0.5 * g
p_lr = 1 / (1 + np.exp(-Ate @ w))
print(f"logistic regression test accuracy {np.mean((p_lr > 0.5) == yte):.3f}")

<details><summary><b>Answer and explanation</b></summary>

Logistic regression gets ≈ 98%. It has the same form of boundary as LDA, but it fits the weights to directly separate the classes, without needing the Gaussian story to be true. With a few hundred samples, being right about less is an advantage (Murphy §9.4).

Expected output:
```
logistic regression test accuracy 0.983
```

</details>

## Milestone 5 · ROC curves and AUC

Accuracy depends on the 0.5 threshold. AUC doesn't: it is the probability that a random malignant case gets a higher score than a random benign one. Compute it for all three models by comparing every malignant/benign pair.

In [ ]:
# Your turn
def auc(score, y):
    pos, neg = score[y == 1], score[y == 0]
    return ...  # TODO: P(random positive outranks random negative)
for name, p in [("naive Bayes", p_nb), ("LDA", p_lda), ("logistic", p_lr)]:
    print(f"{name:<12} AUC {auc(p, yte):.4f}")

In [ ]:
# @title Solution 5 (double-click to show the code)
def auc(score, y):
    pos, neg = score[y == 1], score[y == 0]
    return np.mean((pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :]))  # P(random positive outranks random negative)
for name, p in [("naive Bayes", p_nb), ("LDA", p_lda), ("logistic", p_lr)]:
    print(f"{name:<12} AUC {auc(p, yte):.4f}")

<details><summary><b>Answer and explanation</b></summary>

AUC: naive Bayes 0.982, LDA 0.994, logistic 0.999. All three rank cases far better than accuracy suggests: even naive Bayes almost always scores a malignant case above a benign one. Its accuracy loss comes largely from where its probabilities sit relative to 0.5, not from ranking, which is exactly what the next step examines (Murphy §5.1.3).

Expected output:
```
naive Bayes  AUC 0.9822
LDA          AUC 0.9935
logistic     AUC 0.9987
```

</details>

## Milestone 6 · Calibration: do the probabilities mean what they say?

Use 5-fold cross-validation on the training set to get an out-of-fold probability for every training sample from naive Bayes and from logistic regression. Bin the probabilities into [0, 0.1), …, [0.9, 1]. For each bin, compare the average predicted probability with the actual malignant fraction. Also report the Brier score (mean squared error of the probabilities).

In [ ]:
# Your turn
def lr_fit(A, y, lam=0.01, steps=3000):
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        p = 1 / (1 + np.exp(-A @ w)); w -= 0.5 * (A.T @ (p - y) / len(y) + lam * np.r_[0, w[1:]])
    return w
folds = np.array_split(np.random.default_rng(1).permutation(len(ytr)), 5)
oof_nb, oof_lr = np.zeros(len(ytr)), np.zeros(len(ytr))
for k in range(5):
    v = folds[k]; t = np.concatenate([folds[j] for j in range(5) if j != k])
    oof_nb[v] = nb_proba(nb_fit(Xtr[t], ytr[t]), Xtr[v])
    oof_lr[v] = 1 / (1 + np.exp(-A[v] @ lr_fit(A[t], ytr[t])))
for name, p in [("naive Bayes", oof_nb), ("logistic", oof_lr)]:
    bins = np.minimum((p * 10).astype(int), 9); rows = []
    for k in range(10):
        m = bins == k
        ...  # TODO: predicted vs observed in each bin
    print(f"{name}: Brier {np.mean((p - ytr) ** 2):.4f}")
    print("   " + " | ".join(rows))

In [ ]:
# @title Solution 6 (double-click to show the code)
def lr_fit(A, y, lam=0.01, steps=3000):
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        p = 1 / (1 + np.exp(-A @ w)); w -= 0.5 * (A.T @ (p - y) / len(y) + lam * np.r_[0, w[1:]])
    return w
folds = np.array_split(np.random.default_rng(1).permutation(len(ytr)), 5)
oof_nb, oof_lr = np.zeros(len(ytr)), np.zeros(len(ytr))
for k in range(5):
    v = folds[k]; t = np.concatenate([folds[j] for j in range(5) if j != k])
    oof_nb[v] = nb_proba(nb_fit(Xtr[t], ytr[t]), Xtr[v])
    oof_lr[v] = 1 / (1 + np.exp(-A[v] @ lr_fit(A[t], ytr[t])))
for name, p in [("naive Bayes", oof_nb), ("logistic", oof_lr)]:
    bins = np.minimum((p * 10).astype(int), 9); rows = []
    for k in range(10):
        m = bins == k
        if m.sum(): rows.append(f"{k/10:.1f}: said {p[m].mean():.2f} saw {ytr[m].mean():.2f} (n={m.sum()})")  # predicted vs observed in each bin
    print(f"{name}: Brier {np.mean((p - ytr) ** 2):.4f}")
    print("   " + " | ".join(rows))

<details><summary><b>Answer and explanation</b></summary>

Naive Bayes puts almost every sample in the 0.0 or 0.9 bin: it is extremely sure of itself, because multiplying 30 'independent' likelihoods that are really correlated compounds the overconfidence. When it's wrong, it is confidently wrong (Brier 0.059). Logistic regression's probabilities track reality much better (Brier 0.022): when it says 0.1–0.2 it's right about 14% of the time. Calibration matters whenever someone acts on the number itself, such as a doctor weighing a 20% risk. Ranking quality (AUC) and calibration are different properties.

Expected output:
```
naive Bayes: Brier 0.0586
   0.0: said 0.00 saw 0.04 (n=280) | 0.1: said 0.13 saw 1.00 (n=2) | 0.4: said 0.44 saw 0.50 (n=2) | 0.5: said 0.57 saw 0.50 (n=2) | 0.7: said 0.76 saw 0.50 (n=2) | 0.8: said 0.88 saw 0.00 (n=1) | 0.9: said 1.00 saw 0.93 (n=165)
logistic: Brier 0.0217
   0.0: said 0.02 saw 0.01 (n=249) | 0.1: said 0.14 saw 0.14 (n=21) | 0.2: said 0.23 saw 0.00 (n=9) | 0.3: said 0.35 saw 0.00 (n=8) | 0.4: said 0.45 saw 0.75 (n=4) | 0.5: said 0.54 saw 0.80 (n=5) | 0.6: said 0.65 saw 0.80 (n=5) | 0.7: said 0.75 saw 1.00 (n=5) | 0.8: said 0.85 saw 1.00 (n=7) | 0.9: said 0.99 saw 1.00 (n=141)
```

</details>

## Milestone 7 · Decide: a missed cancer costs 10× a false alarm

Decision theory: predict malignant when P(malignant | x) > threshold. With cost 10 for a false negative and 1 for a false positive, the cost-minimising threshold is 1/(1 + 10) ≈ 0.091 (Murphy §5.1). Using logistic regression on the test set, compare the confusion matrix and total cost at threshold 0.5 with those at 0.091.

In [ ]:
# Your turn
def report(t):
    pred = p_lr > t
    fn = int(np.sum(~pred & (yte == 1))); fp = int(np.sum(pred & (yte == 0)))
    tp = int(np.sum(pred & (yte == 1))); tn = int(np.sum(~pred & (yte == 0)))
    print(f"threshold {t:.3f}: TP {tp:>2} FN {fn:>2} FP {fp:>2} TN {tn:>2}   cost = 10*FN + FP = {10 * fn + fp}")
report(0.5)
...  # TODO: cost-optimal threshold c_FP / (c_FP + c_FN)

In [ ]:
# @title Solution 7 (double-click to show the code)
def report(t):
    pred = p_lr > t
    fn = int(np.sum(~pred & (yte == 1))); fp = int(np.sum(pred & (yte == 0)))
    tp = int(np.sum(pred & (yte == 1))); tn = int(np.sum(~pred & (yte == 0)))
    print(f"threshold {t:.3f}: TP {tp:>2} FN {fn:>2} FP {fp:>2} TN {tn:>2}   cost = 10*FN + FP = {10 * fn + fp}")
report(0.5)
report(1 / (1 + 10))  # cost-optimal threshold c_FP / (c_FP + c_FN)

<details><summary><b>Answer and explanation</b></summary>

At 0.5: 2 missed cancers and 0 false alarms, cost 20. At 0.091: 0 missed cancers and 14 false alarms, cost 14. The same model with a different decision rule gives fewer of the expensive errors. The threshold comes from expected loss: predict malignant when p·10 > (1 − p)·1, i.e. p > 1/11 (Murphy §5.1.2). The model supplies probabilities; the costs decide what to do with them. That's why calibration (step 6) matters: the rule only works if p means what it says.

Expected output:
```
threshold 0.500: TP 41 FN  2 FP  0 TN 72   cost = 10*FN + FP = 20
threshold 0.091: TP 43 FN  0 FP 14 TN 58   cost = 10*FN + FP = 14
```

</details>

## Milestone 8 · Learning curves: who wins with little data?

Train naive Bayes and logistic regression on the first m training samples, for m = 10, 20, 40, 80, 160 and all. Average test accuracy over 20 random orderings. Which model wins with tiny data, and which with plenty?

In [ ]:
# Your turn
for m in [10, 20, 40, 80, 160, len(ytr)]:
    acc_nb, acc_lr = [], []
    for r in range(20):
        idx = np.random.default_rng(100 + r).permutation(len(ytr))[:m]
        if len(set(ytr[idx])) < 2: continue
        acc_nb.append(np.mean((nb_proba(nb_fit(Xtr[idx], ytr[idx]), Xte) > 0.5) == yte))
        acc_lr.append(np.mean(((Ate @ lr_fit(A[idx], ytr[idx], steps=1500)) > 0) = = ...  # TODO: same model, trained on m samples
    print(f"m = {m:>3}: naive Bayes {np.mean(acc_nb):.3f}   logistic {np.mean(acc_lr):.3f}")

In [ ]:
# @title Solution 8 (double-click to show the code)
for m in [10, 20, 40, 80, 160, len(ytr)]:
    acc_nb, acc_lr = [], []
    for r in range(20):
        idx = np.random.default_rng(100 + r).permutation(len(ytr))[:m]
        if len(set(ytr[idx])) < 2: continue
        acc_nb.append(np.mean((nb_proba(nb_fit(Xtr[idx], ytr[idx]), Xte) > 0.5) == yte))
        acc_lr.append(np.mean(((Ate @ lr_fit(A[idx], ytr[idx], steps=1500)) > 0) == yte))  # same model, trained on m samples
    print(f"m = {m:>3}: naive Bayes {np.mean(acc_nb):.3f}   logistic {np.mean(acc_lr):.3f}")

<details><summary><b>Answer and explanation</b></summary>

Logistic regression wins at every size here (e.g. 0.875 vs 0.784 at m = 10, 0.983 vs 0.904 with all data), and naive Bayes plateaus around 91% by m = 20. The textbook result (Ng and Jordan) is that generative models reach their best faster, but that best can be worse. Here the naive Bayes assumption is wrong enough (step 2) that its plateau is low and early. It does stop improving almost immediately, which is the 'fast but limited' half of the story (Murphy §9.4). Always measure: rules of thumb depend on how wrong each model's assumptions are.

Expected output:
```
m =  10: naive Bayes 0.784   logistic 0.875
m =  20: naive Bayes 0.909   logistic 0.913
m =  40: naive Bayes 0.913   logistic 0.930
m =  80: naive Bayes 0.920   logistic 0.950
m = 160: naive Bayes 0.913   logistic 0.960
m = 454: naive Bayes 0.904   logistic 0.983
```

</details>